In [ ]:
from litellm import completion
from dotenv import load_dotenv
import json
# from pricer.batch import Batch
# from pricer.batch_ollama import Batch
# from pricer.batch_ollama import client
from pricer.batch_ollama_threading import Batch
from pricer.batch_ollama_threading import client
from pricer.items import Item

load_dotenv(override=True)

In [ ]:
LITE_MODE = True

In [ ]:
# username = "ed-donner"
username = "mzhu007"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
items[2].id

In [ ]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [ ]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

In [ ]:
!ollama list

In [ ]:
!ollama pull gpt-oss:20b

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
# response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")
response = client.chat.completions.create(messages=messages, model="gpt-oss:20b")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
# print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/gpt-oss:20b", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

In [ ]:
# Batch.create(items, LITE_MODE)
Batch.create(items[1500:1700], LITE_MODE)

In [ ]:
!launchctl getenv OLLAMA_NUM_PARALLEL

In [ ]:
Batch.run()

Simple iteration: single thread, single process

Roughly 10 seconds per call 

36 minutes for 365 items / 1000, there will be 22 batch (22,000)
Estimated time is 22,000 / 365 * 36 = 2,169.863 = 2170 minutes = 36 hours 
If using full data: 820,000 / 22,000 * 36 = 1342 hours = 56 days

Processing 0_1000:  40%
 400/1000 [40:11<57:56,  5.79s/it]

=====

Multi-threading, thread-pool 2

10 minutes for 100 items, roughly 10 * 60 / 100 = 6 seconds per call -- roughly consistent with 2 threads

=====

Multi-threading, thread-pool 10

45 minutes for 500 items, roughly 45 * 60 / 500 = 5.4 seconds per call -- roughly consistent with 2 threads, does not match 10 threads

Finished 1000 calls (100 x 10 baches), took 83m 36s, one call took (83 * 60 + 36) / 1000 = 5.016 seconds


launchctl setenv OLLAMA_NUM_PARALLEL 4
ollama run gpt-oss:20b
MAX_WORKERS = 3
BATCH_SIZE = 100
Batch.create(items[1300:1500], LITE_MODE)
Finished 200 calls (100 x 2 baches), took 11m 58.3s, one call took (11 * 60 + 58) / 200 = 3.59 seconds

ps -ef | grep llama
kill -9 19095
kill -9 19097
launchctl setenv OLLAMA_NUM_PARALLEL 10
launchctl getenv OLLAMA_NUM_PARALLEL 
ollama run gpt-oss:20b
ps -ef | grep llama
See "-np 10"
ollama run gpt-oss:20b
MAX_WORKERS = 10
BATCH_SIZE = 20
Batch.create(items[1500:1700], LITE_MODE)
Finished 200 calls (20 x 10 baches), took 10m 32s, one call took (10 * 60 + 32) / 200 = 3.16 seconds (ollama parable 10, multi-threading 10)
